# Getting EchoCare weights

In [ ]:
import torch
from monai.networks.nets.swin_unetr import SwinTransformer
checkpoint_path = "./echocare_encoder.pth"

: 

In [ ]:
encoder = SwinTransformer(
        in_chans=3,
        embed_dim=128,
        window_size=[8] * 2,
        patch_size=[2] * 2,
        depths=[2, 2, 18, 2],
        num_heads=[4, 8, 16, 32],
        mlp_ratio=4.0,
        qkv_bias=True,
        use_checkpoint=True,
        spatial_dims=2,
        use_v2=True
    )

if checkpoint_path is not None:
        state_dict = torch.load(checkpoint_path, map_location="cpu")
        state_dict.pop("mask_token", None)  # remove if present
        encoder.load_state_dict(state_dict, strict=True)
        print("Loaded EchoCare pretrained weights")
else:
        print("No checkpoint — using random init (not recommended)")

# Check if weights were loaded correctly

In [ ]:
x = torch.rand(1, 3, 256, 256)
x_outs = encoder(x)
print([x_out.shape for x_out in x_outs])

# Load our data

In [ ]:
import re
import os
import pandas as pd
from torch.utils.data import Dataset, DataLoader, random_split
from PIL import Image
from torchvision import transforms as T

def parse_filename(filename):
    # Match x and y values — m prefix means negative
    pattern = r'_x(m?)(\d+)_(\d+)_y(m?)(\d+)_(\d+)'
    match = re.search(pattern, filename)

    if not match:
        print(f"WARNING: Could not parse {filename}, skipping")
        return None

    x_neg, x_int, x_dec, y_neg, y_int, y_dec = match.groups()

    delta_x = float(f"{x_int}.{x_dec}")
    delta_y = float(f"{y_int}.{y_dec}")

    if x_neg == "m":
        delta_x = -delta_x
    if y_neg == "m":
        delta_y = -delta_y

    return delta_x, delta_y

df = pd.DataFrame(columns=["image_path", "delta_x", "delta_y"])
with os.scandir("./Images") as entries:
    for entry in entries:
        delta_x, delta_y = parse_filename(entry.name)
        df.loc[len(df)] = [f"./Images/{entry.name}", delta_x, delta_y]
df.to_csv("dataset.csv", index=False)

class ImageDataset(Dataset):
    def __init__(self, csv_file):
        # Load the CSV file once during initialization
        self.data = pd.read_csv(csv_file)
        self.transform = T.Compose([
                T.Resize((640, 480)),
                T.Grayscale(num_output_channels=1),
                T.ToTensor()
            ])
        
    def __len__(self):
        # Return the total number of samples
        return len(self.data)

    def __getitem__(self, idx):
        # Retrieve a single row at the given index
        # Use .iloc to access the specific row
        sample = self.data.iloc[idx]
        
        # Separate features (X) and label (y)
        # Convert them to tensors (ensure they are numeric types)
        image = Image.open(sample["image_path"])
        image = self.transform(image)
        delta_x, delta_y = sample["delta_x"], sample["delta_y"]
        label = torch.tensor([delta_x, delta_y], dtype = torch.float32)
        
        return image, label

dataset = ImageDataset(csv_file="dataset.csv")
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_data, test_data = random_split(dataset, [train_size, test_size])
train_loader = DataLoader(train_data, batch_size = 4, shuffle = True)
test_loader = DataLoader(test_data, batch_size = 4, shuffle = False)

# Check if data was loaded correctly

In [ ]:
print("Train data length:", len(train_data))
print("Test data length:", len(test_data))
# Pull one batch of data
data_iter = iter(train_loader)
images, labels = next(data_iter)

# Check shapes and types
print(f"Feature batch shape: {images.shape}")
print(f"Labels batch shape: {labels.shape}")
print(f"Data type: {images.dtype}")